# PyTorchLinear CPU training

A single linear layer learns y = 2x + 1. Data preparation, model construction, training, evaluation and checkpoint restoration use the public Bovi contracts.

In [1]:
import os

os.environ["CUDA_VISIBLE_DEVICES"] = "-1"
os.environ["TF_NUM_INTRAOP_THREADS"] = "1"
os.environ["TF_NUM_INTEROP_THREADS"] = "1"
from pathlib import Path
from tempfile import TemporaryDirectory
from uuid import uuid4

from bovi_core.config import Config
from bovi_core.ml import EvaluationContext, ResolvedCheckpoint, TrainingContext
from pytorch_linear import (
    PyTorchLinearEvaluationConfig,
    PyTorchLinearEvaluator,
    PyTorchLinearModelConfig,
    PyTorchLinearModelProvider,
    PyTorchLinearTrainer,
    PyTorchLinearTrainingConfig,
    create_dataloader,
)

Config.reset()
config = Config(experiment_name="pytorch_linear", project_name="pytorch-linear")
model_config = PyTorchLinearModelConfig.from_config(config)
training_config = PyTorchLinearTrainingConfig.from_config(config)
loaders = {
    split: create_dataloader(config, model_config, split) for split in ("train", "validation")
}
print(model_config)
print(next(iter(loaders["train"])))

🔄 Config singleton reset
🔧 Initializing Config...
   🌍 Environment: local
🔍 No TOML path passed, resolving path automatically
🔍 Found pyproject.toml in parent directory: /home/douwe/work/code/bovi-analytics/bovi/.worktrees/model-package-layout/packages/models/pytorch-linear
      📋 Project name: pytorch-linear
      📁 Current working directory: /home/douwe/work/code/bovi-analytics/bovi/.worktrees/model-package-layout/packages/models/pytorch-linear/notebooks/experiments/pytorch_linear
      📁 Traversed to: /home/douwe/work/code/bovi-analytics/bovi/.worktrees/model-package-layout/packages/models/pytorch-linear
      📁 project_src: /home/douwe/work/code/bovi-analytics/bovi/.worktrees/model-package-layout/packages/models/pytorch-linear/src
⚠️  .env file not specified or not found at: /home/douwe/work/code/bovi-analytics/bovi/.worktrees/model-package-layout/packages/models/pytorch-linear/.env
   👤 Author: Douwe de Kok <douwedekok@gmail.com>
🔍 No config_file_path passed, resolving from:
    

In [2]:
workspace = TemporaryDirectory(prefix="pytorch-linear-")
output = Path(workspace.name)
provider = PyTorchLinearModelProvider()
model = provider.create(model_config)
context = TrainingContext(run_id=uuid4(), output_dir=output / "first")
result = PyTorchLinearTrainer(model, loaders, training_config, context).train()
assert result.status == "completed", result.issues
print(result.status, result.stop_reason)
print("First epoch:", result.epochs[0].metrics)
print("Last epoch:", result.epochs[-1].metrics)
assert result.epochs[-1].metrics["train_mse"] < 0.001
print("Best epoch:", result.best_epoch)

completed max_epochs_reached
First epoch: {'train_mse': 1.5945777893066406, 'train_mae': 1.063295841217041, 'validation_mse': 1.5450897216796875, 'validation_mae': 1.0299804210662842}
Last epoch: {'train_mse': 5.804751523896812e-08, 'train_mae': 0.00021868199110031128, 'validation_mse': 5.575902761734142e-08, 'validation_mae': 0.00020994991064071655}
Best epoch: 40


In [3]:
evaluation = PyTorchLinearEvaluator(
    model, PyTorchLinearEvaluationConfig.from_config(config)
).evaluate(
    loaders["validation"],
    EvaluationContext(
        evaluation_id=uuid4(),
        split="validation",
        model_version="last",
        training_run_id=context.run_id,
        output_dir=output / "evaluation",
    ),
)
assert evaluation.status == "completed", evaluation.issues
print(evaluation.metrics)
print("Prediction at x=0.5 (expected 2):", model([[0.5]]))

{'mse': 5.575902761734142e-08, 'mae': 0.00020994991064071655}
Prediction at x=0.5 (expected 2): [1.9998507]


## Resume

Load the last checkpoint into a new model. A new attempt gets its own run ID and starts at epoch 1. Best and last checkpoints stay on disk, while results contain references.

In [4]:
reference = result.last_checkpoint
restored = provider.restore_checkpoint(
    model_config,
    ResolvedCheckpoint(
        format=reference.format,
        source_uri=reference.uri,
        local_path=context.output_dir / "checkpoints" / "last.pt",
    ),
)
resume_context = TrainingContext(
    run_id=uuid4(), resumed_from_run_id=context.run_id, output_dir=output / "resume"
)
resumed = PyTorchLinearTrainer(
    restored, loaders, PyTorchLinearTrainingConfig(epochs=2), resume_context
).train()
assert resumed.status == "completed", resumed.issues
assert resumed.epochs[0].epoch == 1
print(resumed.epochs[-1].metrics)
workspace.cleanup()
Config.reset()

{'train_mse': 2.434527779371365e-08, 'train_mae': 0.0001416131854057312, 'validation_mse': 2.3388111003441736e-08, 'validation_mae': 0.0001359805464744568}
🔄 Config singleton reset
